## Show, Attend and Tell — Training (Flickr8k)


Clone repository

In [ ]:
!git clone https://github.com/seanzhangw/show-attend-tell.git

Pull recent changes

In [ ]:
%cd /content/show-attend-tell
!git pull

Get dataset

In [ ]:
%cd /content/show-attend-tell/data
!python get_flickr8k.py

In [ ]:
%cd /content/show-attend-tell/code

import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

import config
from datasets.flickr8k import collate_fn, build_flickr8k_dataset_split
from eval.greedy_decode import greedy_decode
from models.encoder import EncoderCNN
from models.decoder import Decoder


Training, validation functions

In [ ]:
from training.loop import compute_caption_loss, train_one_epoch, validate

Define Hyperparameters

In [ ]:
# Hyperparameters (edit as needed)
epochs = 8
batch_size = config.BATCH_SIZE
lr = 1e-4
embed_dim = 512
hidden_dim = 512
attention_dim = 512

dropout = 0.5
lambda_att = 1.0
grad_clip = 5.0
freeze_encoder = True
val_ratio = 0.1
split_seed = 42
# BLEU-4: cap images per eval for speed (None = all val images)
bleu_max_images = 500

Build datasets, dataloaders

In [ ]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            # ImageNet normalization for pre-trained ResNet (ImageNet dataset mean and std)
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

# Dataset: image-level train/val split; vocab from train captions only
(
    train_set,
    val_set,
    val_image_ids,
    full_captions_map,
    word2idx,
    idx2word,
) = build_flickr8k_dataset_split(
    config,
    transform=transform,
    val_ratio=val_ratio,
    seed=split_seed,
)
pad_idx = word2idx["<pad>"]
print(
    f"Train samples: {len(train_set)} | Val samples: {len(val_set)} | "
    f"Val images: {len(val_image_ids)} | Vocab size: {len(word2idx)}"
)

train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_set,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
)

Initialize models, loss, and optimizer

In [ ]:
# Models
encoder = EncoderCNN().to(device)
decoder = Decoder(
    vocab_size=len(word2idx),
    embed_dim=embed_dim,
    feature_dim=2048,
    hidden_dim=hidden_dim,
    attention_dim=attention_dim,
    dropout=dropout,
).to(device)

# Loss / Optimizer
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

params = list(decoder.parameters())
if not freeze_encoder:
    params += [p for p in encoder.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=lr)

Training

In [ ]:
# =========================
# 5b) Train + best checkpoint (metrics + resume)
# =========================

from training.loop import fit

CHECKPOINT_DIR = "./checkpoints"
RESUME_PATH = None  # e.g. "./checkpoints/best_by_val_loss.pt"
EVAL_METRICS = ["bleu1", "bleu2", "bleu3", "bleu4"]  # add "meteor" after nltk.download(...)
BEST_BY = "val_loss"
BEST_MODE = "min"  # "max" when BEST_BY is a score to maximize (e.g. "bleu4")

_, best_ckpt_path, _ = fit(
    encoder,
    decoder,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs,
    val_image_ids,
    full_captions_map,
    transform,
    word2idx,
    idx2word,
    config.IMAGE_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    eval_metric_names=EVAL_METRICS,
    best_by=BEST_BY,
    best_mode=BEST_MODE,
    resume_path=RESUME_PATH,
    max_decode_len=config.MAX_LEN,
    bleu_max_images=bleu_max_images,
    lambda_att=lambda_att,
    grad_clip=grad_clip,
    print_every=100,
    freeze_encoder=freeze_encoder,
    hparams={
        "embed_dim": embed_dim,
        "hidden_dim": hidden_dim,
        "attention_dim": attention_dim,
        "dropout": dropout,
        "lambda_att": lambda_att,
        "freeze_encoder": freeze_encoder,
        "lr": lr,
        "batch_size": batch_size,
        "val_ratio": val_ratio,
        "split_seed": split_seed,
    },
)


In [ ]:
from PIL import Image

num_examples = min(20, len(val_image_ids))
print(f"\nQualitative eval on {num_examples} validation images (unique)")

for i, img_name in enumerate(sorted(val_image_ids)[:num_examples]):
    path = os.path.join(config.IMAGE_DIR, img_name)
    image = Image.open(path).convert("RGB")
    image_tensor = transform(image)

    pred_tokens = greedy_decode(
        encoder,
        decoder,
        image_tensor,
        word2idx,
        idx2word,
        device,
        max_len=config.MAX_LEN,
    )
    refs = full_captions_map[img_name][:5]

    print(f"\n[{i+1:02d}] Image: {img_name}")
    print("Generated:", " ".join(pred_tokens) if pred_tokens else "<empty>")
    for j, ref in enumerate(refs[:3], start=1):
        print(f"Ref {j}: {ref}")


Save the trained model to Google Drive

In [ ]:
from google.colab import drive
import sys

drive.mount('/content/gdrive')

# NOTE: Change this path to your own Google Drive path
base_dir = "/content/gdrive/MyDrive/show-attend-tell"
sys.path.append(base_dir)

In [ ]:
import shutil
import os

# Define the source (where the model is now) and destination (your Drive)
source_path = './checkpoints/best_by_val_loss.pt'
dest_path = os.path.join(base_dir, 'best_by_val_loss.pt')

# Create the directory on Drive if it doesn't exist
os.makedirs(base_dir, exist_ok=True)

# Copy the file
shutil.copyfile(source_path, dest_path)

print(f"Model successfully saved to: {dest_path}")